In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fs = 1000
dur = 1
f_usf = 5
t = np.linspace(0, dur, fs)
signal = np.sin(2*np.pi*f_usf*t)

plt.plot(t, signal)
plt.grid(alpha=0.5)
plt.title('My first signal')
plt.xlabel('Time, s')
plt.ylabel('Amplitude, mV')
plt.axhline(y=0, linestyle='--', color='r', alpha=0.3)

In [ ]:
import pandas as pd
import scipy.signal as sp

fs = 1000
dur = 1
f_usf = 8
f_intf = 50

t = np.linspace(0,dur,int(fs * dur), endpoint=False)
amp1 = 1.2
amp2 = 1.3
amp3 = 0.2

s_1 = amp1 * np.sin(2*np.pi*f_usf*t)
s_2 = amp2 * np.sin(2*np.pi*f_intf*t)
s_3 = amp3 * np.random.randn(fs*dur)
s_ful = s_1 + s_2 + s_3
df = pd.DataFrame({'raw': s_ful, 'time': t})


a, b = sp.butter(4, 20, 'low', fs=fs)
fn4 = sp.filtfilt(a, b, s_ful)
c, d = sp.iirnotch(50, 30, fs)
clear = sp.filtfilt(c, d, fn4)
df['filtered'] = clear

v_max = np.max(df['filtered'])
v_min = np.min(df['filtered'])
v_mean = np.mean(df['filtered'])
peaks, _ = sp.find_peaks(clear, height=0.5)
peak_t = t[peaks]
periods = np.diff(peak_t)
mp = np.mean(periods)
freq = 1/mp
stdp = np.std(periods)

print(f'Maximal voltage: {v_max} mV')
print(f'Minimal voltage: {v_min} mV')
print(f'Mean voltage: {v_mean} mV')
print(f'Useful frequency: {freq} Hz')
print(f'Standart deviation: {stdp} s')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12,8))
ax1.plot(df['time'], df['raw'], linewidth=0.8, color='b', alpha=0.5, label='Raw signal')
ax1.set_title('Signals')
ax1.set_xlabel('Time, s')
ax1.set_ylabel('Voltage, mV')
ax1.grid()
ax1.axhline(y=0, linestyle='--', linewidth=0.8, alpha=0.6, color='r')
ax1.plot(df['time'], df['filtered'], linewidth=1, color='k', label='Filtered signal')
ax1.plot(t[peaks], clear[peaks], 'o', color='orange', label='Peaks') 
ax1.legend()

ax2.plot(df[df['time'] <= 0.1]['time'], df[df['time'] <= 0.1]['raw'], linewidth=0.7, color='g', label='Raw signal')
ax2.set_title('Signals within 0,1 sec')
ax2.set_xlabel('Time, s')
ax2.set_ylabel('Voltage, mV')
ax2.grid()
ax2.axhline(y=0, linestyle='--', linewidth=0.8, alpha=0.6, color='r')
ax2.plot(df[df['time'] <= 0.1]['time'], df[df['time'] <= 0.1]['filtered'], linewidth=1, color='k', label='Filtered signal')
ax2.legend()

fig.subplots_adjust(hspace=0.5)